# Final Project - Kansas HMDA Home-Purchase Applications
Scott Person

## Project Overview

Among Kansas home-purchase applications that received a lending decision in 2018–2025, were denial rates higher in lower-income census tracts than in tracts at or above the area median, even when comparing applicants within similar income bands?

The unit is an application record, not a unique borrower. A denial is `action_taken == 3`; the denominator is applications with a lending decision (`action_taken` 1, 2, or 3). Withdrawn, incomplete, and other non-decision records do not belong in that denominator. Income is reported in thousands of dollars; the tract-income percentage is relative to an area median, not the applicant's income. The four applicant-income bands are under $50,000, $50,000–<$100,000, $100,000–<$150,000, and $150,000 or more.

Source: [FFIEC HMDA Data Browser](https://ffiec.cfpb.gov/data-browser/), Kansas home-purchase records for 2018–2025. See the [HMDA action-taken field definitions](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields/#action_taken). The accompanying `data/analysis_data.csv` is a seven-column extract of the eight annual files in this folder. `Download_KS_Data.ipynb` records the retrieval process. Rates here describe recorded applications, not the probability that an otherwise identical applicant would be denied.

## Initial Data Review

The extract retains the lending action, tract income relative to area median, reported applicant income, year, metropolitan-area code, state, and loan purpose. This smaller analysis file is sufficient for the stated question; the annual source files are retained separately. The notebook expects to run from this folder, with the CSV under `data/`.

In [0]:
%pip install seaborn # Databricks does not install seaborn by default

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('data/analysis_data.csv')
raw = pd.read_csv(DATA_PATH).rename(columns={'derived_msa-md': 'derived_msa_md'})
required = {'action_taken', 'tract_to_msa_income_percentage', 'income',
            'activity_year', 'derived_msa_md', 'state_code', 'loan_purpose'}
assert required.issubset(raw.columns), f'Missing columns: {required - set(raw.columns)}'
print(f'{len(raw):,} application records; {len(raw.columns)} columns')
print(raw.dtypes.to_string())
print(raw.head(3).to_string(index=False))

In [0]:
print('Numeric summaries (income is in $1,000s):')
print(raw[['income', 'tract_to_msa_income_percentage', 'activity_year']].describe().round(2).to_string())
print('\nRecords by year:')
print(raw['activity_year'].value_counts().sort_index().to_string())
print('\nAction codes:')
print(raw['action_taken'].value_counts().sort_index().to_string())

## Data Integrity Audit

### Shape, columns, and data types

There is no application identifier in this seven-column extract. Year, action, loan purpose, and metropolitan-area code are coded fields; treating their codes as continuous measurements would be misleading. The source has one state and one loan purpose by design. The decision-rate denominator is limited to actions 1–3, not all rows.

In [0]:
print('State codes:', raw['state_code'].value_counts(dropna=False).to_dict())
print('Loan-purpose codes:', raw['loan_purpose'].value_counts(dropna=False).to_dict())
print('Metropolitan-area codes:', raw['derived_msa_md'].nunique())
print('Non-metropolitan code 99999:', int(raw['derived_msa_md'].eq(99999).sum()))

### Missing values

The table distinguishes actual nulls from literal text placeholders and zero. Zero is a valid value in some columns, so it is a flag for investigation, not a universal missing-value rule. An absent applicant income prevents assigning an income band; zero reported income does not. A zero tract-income percentage cannot support the intended below/above-median comparison and is excluded below.

In [0]:
audit_rows = []
for column in raw.columns:
    values = raw[column].astype('string').str.strip()
    lower = values.str.lower()
    audit_rows.append({
        'column': column,
        'null': int(raw[column].isna().sum()),
        'blank_text': int(values.eq('').sum()),
        'literal_na': int(lower.eq('na').sum()),
        'question_mark': int(values.eq('?').sum()),
        'exempt': int(lower.eq('exempt').sum()),
        'zero': int(pd.to_numeric(values, errors='coerce').eq(0).sum()),
    })
missing_audit = pd.DataFrame(audit_rows).fillna(0)
print(missing_audit.to_string(index=False))

### Duplicates

Identical rows across seven broad fields need not be duplicate applications: different applicants can share these attributes. The extract has no unique application identifier, so it cannot establish whether a repeated combination represents a true duplicate. Retain these rows rather than dropping valid applications or understating rates. A future source-level audit would require a suitable identifier or additional fields.

In [0]:
print('Identical seven-field rows after the first:', int(raw.duplicated().sum()))
print('Application identifier in extract:', 'not present')

### Validity checks and outliers

Check the allowed action codes, Kansas/state and home-purchase/loan-purpose scope, year range, and tract-income percentage. The `99999` metropolitan-area code identifies a non-metropolitan segment; do not mistake it for a numeric outlier or silently mix its interpretation with named metro areas. Extreme applicant income merits inspection, but HMDA income is in thousands and may include zero or negative values. No income is capped or deleted solely for being extreme: the analysis uses bands, not a mean income, and such values fall into the boundary bands. This preserves coverage while limiting their influence on the reported statistic.

In [0]:
print('Unexpected actions:', int((~raw['action_taken'].isin(range(1, 9))).sum()))
print('Unexpected state:', int(raw['state_code'].ne('KS').sum()))
print('Unexpected loan purpose:', int(raw['loan_purpose'].ne(1).sum()))
print('Outside 2018–2025:', int((~raw['activity_year'].between(2018, 2025)).sum()))
print('Tract income <= 0:', int(raw['tract_to_msa_income_percentage'].le(0).sum()))
print('Missing tract income:', int(raw['tract_to_msa_income_percentage'].isna().sum()))
print('Negative incomes:', int(raw['income'].lt(0).sum()))
print('Incomes above $1m (income > 1000 in $1,000s):', int(raw['income'].gt(1000).sum()))
print('Income min/max ($1,000s):', raw['income'].min(), raw['income'].max())

## Cleaning & Remediation

The filters below make the denominator match the question. They operate on a copy and log both the number removed and the denials removed; exclusions change the population and may change the rate. No rows are removed merely because the seven-field combination repeats. No extreme reported incomes are winsorized or imputed. The separate metropolitan/non-metropolitan comparison below checks whether pooling those segments obscures the tract-group pattern.

In [0]:
def denial_rate(frame):
    return 100 * frame['action_taken'].eq(3).mean() if len(frame) else np.nan

work = raw.copy()
steps = []
def retain(label, mask):
    global work
    before = work
    work = before.loc[mask].copy()
    steps.append({
        'decision': label,
        'before': len(before),
        'removed': len(before) - len(work),
        'removed_denials': int(before['action_taken'].eq(3).sum() - work['action_taken'].eq(3).sum()),
        'remaining': len(work),
        'denial_rate_pct': denial_rate(work),
    })

retain('Decisions only: actions 1, 2, 3', work['action_taken'].isin([1, 2, 3]))
retain('Positive tract-income percentage', work['tract_to_msa_income_percentage'].gt(0))
retain('Reported applicant income', work['income'].notna())
clean = work
log = pd.DataFrame(steps)
print(log.round(2).to_string(index=False))
assert clean['action_taken'].isin([1, 2, 3]).all()
assert clean['tract_to_msa_income_percentage'].gt(0).all()
assert clean['income'].notna().all()

Decision log and assumptions:

- Actions outside 1–3 did not receive a lending decision. Excluding them defines the outcome rather than treating them as approvals. Their removal changes the denominator, so an all-action “raw denial rate” is not directly comparable to the decided-application rate.
- A zero or missing tract-income percentage cannot be classified meaningfully for this question. The excluded records may differ from those retained; we cannot generalize the tract comparison to them.
- Missing applicant income prevents assignment to a band. Dropping these applications may bias band-specific rates if income reporting is related to denial. The log shows exactly how many applications and denials are lost; the finding applies to records with reported income.
- Repeated seven-field combinations and unusual but permitted incomes are retained. This is a deliberate judgment, not proof that every source record is error-free.
- `derived_msa_md == 99999` is kept in the statewide result and shown separately below. Its tract-income reference may differ from a named MSA; compare named-metro results before interpreting the statewide pattern as an MSA-specific effect.

In [0]:
labels = ['Under $50k', '$50k–<$100k', '$100k–<$150k', '$150k+']
clean['income_band'] = pd.cut(clean['income'],
    bins=[-np.inf, 50, 100, 150, np.inf], right=False, labels=labels)
clean['tract_group'] = np.where(clean['tract_to_msa_income_percentage'] < 100,
    'Below median', 'At or above median')
clean['area_type'] = np.where(clean['derived_msa_md'].eq(99999),
    'Non-metropolitan code 99999', 'Named metropolitan area')
assert clean['income_band'].notna().all()

def rates(frame, group_cols):
    result = frame.groupby(group_cols, observed=True)['action_taken'].agg(
        applications='size', denials=lambda actions: actions.eq(3).sum()).reset_index()
    result['denial_rate_pct'] = 100 * result['denials'] / result['applications']
    return result

by_band = rates(clean, ['income_band', 'tract_group'])
print('Statewide decided applications with reported income and positive tract percentage:')
print(by_band.round(2).to_string(index=False))
print('\nSensitivity: same calculation by area type:')
print(rates(clean, ['area_type', 'income_band', 'tract_group']).round(2).to_string(index=False))

### Clean-data handoff to exploratory analysis

`clean` is the analysis-ready application table; `by_band` is the statewide summary with counts, denials, and rates. The area-type table is a sensitivity check, not a second independent dataset. The sections below use the same denominator throughout. Annual rates may shift with the mix of applications and reporting; describe them rather than attributing a causal change to year alone.

## Exploratory Data Analysis & Visualizations

The descriptive rate table above answers the central comparison while retaining its denominators. The first chart shows the tract difference within each applicant-income band. The second shows how each comparison varies across years; separate panels avoid eight overlapping lines. Both use the cleaned application population.

In [0]:
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=by_band, x='income_band', y='denial_rate_pct',
            hue='tract_group', order=labels, errorbar=None, ax=ax)
ax.set(title='Denial rates are higher in below-median tracts within every income band',
       xlabel='Applicant income band', ylabel='Denied applications (%)')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)
ax.legend(title='Tract income relative to area median')
fig.tight_layout()
plt.show()

In [0]:
by_year = rates(clean, ['activity_year', 'income_band', 'tract_group'])
g = sns.relplot(data=by_year, x='activity_year', y='denial_rate_pct',
                hue='tract_group', col='income_band', col_wrap=2, kind='line',
                marker='o', facet_kws={'sharey': False}, height=3.3, aspect=1.35)
g.set_axis_labels('Year', 'Denied applications (%)')
g.set_titles('{col_name}')
g.fig.suptitle('Annual tract-group denial rates within applicant-income bands', y=1.03)
plt.show()
print('Smallest annual group:', int(by_year['applications'].min()), 'applications')

## Key Insights / Conclusions

In this extract, below-median tracts have higher recorded denial rates in every applicant-income band. The statewide rates for applicants reporting $50,000–<$100,000 are about 9.95% versus 7.15%; for $150,000 or more they are about 6.20% versus 3.87%. These figures compare applications with decisions and reported income, not all inquiries or all applicants. Read the group counts beside the rates, and use the annual panels to check whether the pooled result conceals changes over time.

The named-metropolitan-only sensitivity check retains the direction of the pooled comparison in all four income bands. The annual view is less uniform: in the under-$50,000 band the below-median tract rate is slightly lower in 2022 and 2025, so the pooled finding should not be claimed for every year. The non-metropolitan segment has different baseline rates, so the statewide rate should not be described as a result for named MSAs alone. The charts document a descriptive association; they do not tell us why a lender denied an application.

## Limitations & Next Steps

The extract omits credit history, debt-to-income ratio, loan terms, property details, lender, and other factors that could explain differences within broad income bands. Multiple rows may refer to the same borrower, and the seven fields cannot prove whether identical rows are duplicates. Missing applicant income and zero tract percentages remove applications from the analysis; the decision log quantifies their cost. The tract-income percentage for the non-metropolitan code deserves confirmation against HMDA documentation before presenting the statewide grouping as a uniform area-median measure.

For a stronger follow-up, verify the non-metropolitan field definition, test narrower applicant-income bands and year-specific group sizes, and return to the source files for additional underwriting and geography fields. Keep the conclusion descriptive unless those factors can be addressed.